# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jasleen13/ML-Internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess

REPO_URL = "https://github.com/jasleen13/ML-Internship.git"
DATA_REL = Path("data") / "raw" / "content_refresh_anonymized.csv"

def find_repo_root(start: Path):
    p = start
    while not (p / DATA_REL).exists() and p != p.parent:
        p = p.parent
    return p if (p / DATA_REL).exists() else None

repo_root = find_repo_root(Path.cwd())
if repo_root is None:
    clone_dir = Path("/content/ML-Internship") if Path("/content").exists() else Path.cwd() / "ML-Internship"
    if not (clone_dir / DATA_REL).exists():
        subprocess.run(["git", "clone", REPO_URL, str(clone_dir)], check=True)
    repo_root = clone_dir

DATA_PATH = repo_root / DATA_REL
OUTPUTS_DIR = repo_root / "work" / "outputs"
FIGURES_DIR = repo_root / "work" / "figures"
OUTPUTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print("Using data at:", DATA_PATH)


Using data at: /home/claude/ML-Internship/data/raw/content_refresh_anonymized.csv


## 1. Ranked actions + reason codes

**The queue** is the same Week 4 baseline score, unchanged, this notebook turns it into
something a human reviewer can actually act on. Every row keeps one reason code
(`low_ctr_visible_page`) and gets one of two suggested actions, chosen by search intent rather
than a single one-size-fits-all fix, closer to a light segment-to-action mapping than a formal
archetype system (formal clustering into archetypes is Lane 3's job, not this lane's).

**Segment to action mapping:**

| Segment (`main_intent`) | Suggested action | Why this action, not the other |
|---|---|---|
| `informational` | `rewrite_title_meta` | the title/snippet is usually the whole pitch, no purchase signal to add |
| `transactional` | `improve_snippet_signals` | price, availability, or rating signals in the snippet often move clicks more than title wording alone |
| `commercial` | `improve_snippet_signals` | same logic, comparison/review intent responds to trust and pricing signals in the snippet |
| `navigational` | `rewrite_title_meta` | rare in this pool (13 pages), defaults to the simpler fix |

**The decay/refresh insight, stated carefully:** Week 4 checked whether content staleness
(`days_since_last_update >= 180`) predicts underperformance and got a MIXED verdict, the
direction matched the hypothesis only after a volume floor dropped the sample to n=17, too thin
to trust. Staleness is therefore **not** part of this ranking rule, and it's listed as a
monitoring trigger in section 4 rather than an action driver here.


In [2]:
visible = df["impressions_90d"] >= 500
in_range = (df["avg_position"] > 0) & (df["avg_position"] <= 20)
lane = df[visible & in_range].copy()

tier_median_ctr = lane.groupby("position_tier")["ctr"].transform("median")
lane["ctr_gap"] = lane["ctr"] - tier_median_ctr
lane["score"] = (lane["ctr_gap"] < 0).astype(int) * (-lane["ctr_gap"]).clip(lower=0) * np.log1p(lane["impressions_90d"])
lane["reason_code"] = "low_ctr_visible_page"

def action_for(row):
    if row["main_intent"] in ("transactional", "commercial"):
        return "improve_snippet_signals"
    return "rewrite_title_meta"

lane["action"] = lane.apply(action_for, axis=1)
queue = lane.sort_values("score", ascending=False).reset_index(drop=True)

print(f"Ranked queue: {len(queue):,} candidate pages")
print()
print("Action mix across the full queue:")
print(queue["action"].value_counts())
print()
print("Top 10:")
print(queue.head(10)[["content_id", "main_intent", "position_tier", "impressions_90d",
                        "ctr", "ctr_gap", "score", "reason_code", "action"]].to_string(index=False))


Ranked queue: 12,023 candidate pages

Action mix across the full queue:
action
rewrite_title_meta         7252
improve_snippet_signals    4771
Name: count, dtype: int64

Top 10:
          content_id   main_intent position_tier  impressions_90d  ctr  ctr_gap    score          reason_code                  action
content_c8e9d6ab9013 informational        page_1           208678 0.00    -0.24 2.939653 low_ctr_visible_page      rewrite_title_meta
content_453722754fea informational        page_1           140079 0.01    -0.23 2.725493 low_ctr_visible_page      rewrite_title_meta
content_39881853ef0c informational        page_1           112434 0.01    -0.23 2.674930 low_ctr_visible_page      rewrite_title_meta
content_c84a0ab98e90 informational        page_1           223271 0.03    -0.21 2.586391 low_ctr_visible_page      rewrite_title_meta
content_0919dd345d80 informational        page_1           119217 0.02    -0.22 2.571516 low_ctr_visible_page      rewrite_title_meta
content_d274ac4158

## 2. Intended use and limits

**Intended use:** an SEO content editor works down this queue, highest score first, to decide
which pages to open and review for a title, meta description, or snippet rewrite, given limited
review time each week. It is decision-support: it says where to look first, not what to change
or how much it will help.

**Limits:**
- The label is a current-window proxy (a page sits below its tier's median CTR right now), not
  a forecast, it does not claim a rewrite will improve anything, that needs a controlled
  before/after test this dataset can't provide.
- Built and validated on one lane, one dataset (the starter CSV, 28 clients), it has not been
  checked against the full warehouse or any other client base.
- Only covers pages meeting the volume and position floor (`impressions_90d >= 500`,
  `avg_position` 1 to 20), pages outside that range are not scored at all, not confirmed fine.
- Staleness is excluded from the score (see section 1), so a page that's both low-CTR and very
  old gets no extra weight for that, by design, not by oversight.


In [3]:
print(f"Candidate pool covered by this playbook: {len(lane):,} of {len(df):,} total pages ({len(lane)/len(df):.1%})")
print(f"Distinct clients represented: {lane['client_id'].nunique()}")
print(f"Base rate (share flagged is_underperforming): {(lane['ctr_gap'] < 0).mean():.3f}")


Candidate pool covered by this playbook: 12,023 of 30,000 total pages (40.1%)
Distinct clients represented: 28
Base rate (share flagged is_underperforming): 0.490


## 3. Human review + the no-go list

**Before acting on any page in this queue, a reviewer checks:**
1. Does the page actually resolve and get indexed? (catches tracking artifacts before they waste review time)
2. Is a search-results feature likely sitting above the organic result at this position? (a title rewrite can't fix that)
3. Does the page have real word count and content, or is `word_count` missing/zero? (a data gap, not a content problem)
4. Is this page part of a client-wide pattern (same client, same problem, multiple pages)? A shared template fix may beat editing pages one at a time.

**No-go list, what should NOT be automated:**
- **Zero-click extreme outliers should never be auto-flagged as content problems.** 1,215 pages
  in this queue have exactly 0 clicks despite real impression volume, the single highest-scoring
  page in the whole queue combines 208,678 impressions with 0 clicks, that combination is exactly
  where a tracking or indexing bug hides, not a title problem, it needs a manual check first,
  never a straight-to-rewrite pipeline.
- **Pages at position 3 to 4 should not be auto-rewritten without a SERP check.** 736 pages sit
  in this band, close enough to the top 3 that a featured snippet or "people also ask" panel
  could be suppressing clicks in a way this dataset has no visibility into, no title change fixes
  that.
- **No automatic bulk rewrites from this score alone.** The score ranks review priority, it is
  not a QA gate, and it should never trigger a rewrite without a human opening the page first.
- **No archetype-level claims from thin cells.** Any segment cut with n<30 (3 such
  position-tier x content-type cells exist in this data) should be reported as inconclusive, not
  averaged into a headline number.


In [4]:
zero_click = lane[lane["clicks_90d"] == 0]
print(f"Zero-click pages in the queue (no-go for auto-rewrite): {len(zero_click):,}")
print(zero_click[["content_id", "impressions_90d", "avg_position"]].sort_values("impressions_90d", ascending=False).head(3).to_string(index=False))
print()

near_top3 = lane[(lane["avg_position"] >= 3) & (lane["avg_position"] <= 4)]
print(f"Pages at position 3-4, SERP-feature risk zone (no-go without manual SERP check): {len(near_top3):,}")
print()

missing_wc = lane["word_count"].isna().sum()
print(f"Pages with missing word_count (verify content exists before rewriting): {missing_wc:,} of {len(lane):,}")


Zero-click pages in the queue (no-go for auto-rewrite): 1,215
          content_id  impressions_90d  avg_position
content_c8e9d6ab9013           208678           9.7
content_ae6d1339904d            17622          19.5
content_825a9788af8d            16786           5.6

Pages at position 3-4, SERP-feature risk zone (no-go without manual SERP check): 736

Pages with missing word_count (verify content exists before rewriting): 3,662 of 12,023


## 4. Monitoring / retrain triggers

**What would tell this playbook has gone stale:**
- **Candidate pool size shifts sharply** (a new client onboarded, or an existing client's
  tracking changes), the current pool is 12,023 pages across 28 clients, a jump or drop of more
  than roughly 20% in either count is worth a manual look before trusting the queue as-is.
- **Base rate drifts far from today's 0.490.** If the share of pages flagged
  `is_underperforming` moves substantially (say, past 0.60 or below 0.35), the tier medians the
  score is built on may no longer represent current search behavior, and the tier baselines
  should be recomputed.
- **Staleness gets enough volume to actually test.** The decay signal sits at MIXED with n=17,
  once the volume-floored stale-page sample reaches something closer to n=50, it's worth
  rerunning that check, it may finally move to CONFIRMED or FALSE.
- **The Week 5/6 model's honest ROC AUC drops toward the dummy floor (0.500)** on a fresh
  client-grouped split, that would mean the engagement-rate signal the model leans on has
  stopped predicting underperformance, and the model needs retraining or retiring, not just
  rerunning.


In [5]:
print("Monitoring baseline, recorded at this run:")
print(f"  Candidate pool size: {len(lane):,} pages, {lane['client_id'].nunique()} clients")
print(f"  Base rate (is_underperforming): {(lane['ctr_gap'] < 0).mean():.3f}")
print(f"  Staleness signal sample size at volume floor (from Week 4): n=17 (MIXED verdict, watch for n>=50)")
print(f"  Week 5/6 honest model ROC AUC (client-grouped split): 0.727 (retrain trigger: falls toward 0.500)")


Monitoring baseline, recorded at this run:
  Candidate pool size: 12,023 pages, 28 clients
  Base rate (is_underperforming): 0.490
  Staleness signal sample size at volume floor (from Week 4): n=17 (MIXED verdict, watch for n>=50)
  Week 5/6 honest model ROC AUC (client-grouped split): 0.727 (retrain trigger: falls toward 0.500)


## 5. Exports for the paper

The ranked queue (CSV, gitignored by design, regenerated every run) and two committed artifacts
the paper builds on directly: a metrics JSON (the receipt) and a figure showing the action mix
across the queue.


In [6]:
import json

csv_path = OUTPUTS_DIR / "action_playbook_queue.csv"
queue[["content_id", "client_id", "main_intent", "position_tier", "impressions_90d",
       "avg_position", "ctr", "ctr_gap", "score", "reason_code", "action"]].to_csv(csv_path, index=False)
print(f"Wrote ranked queue to: {csv_path}")

metrics = {
    "lane": "Lane 4: CTR / Engagement Opportunity Scoring",
    "candidate_pool_size": int(len(lane)),
    "n_clients": int(lane["client_id"].nunique()),
    "base_rate_is_underperforming": float((lane["ctr_gap"] < 0).mean()),
    "reason_code": "low_ctr_visible_page",
    "actions": {
        "rewrite_title_meta": int((queue["action"] == "rewrite_title_meta").sum()),
        "improve_snippet_signals": int((queue["action"] == "improve_snippet_signals").sum()),
    },
    "no_go_counts": {
        "zero_click_pages": int((lane["clicks_90d"] == 0).sum()),
        "serp_feature_risk_zone_pos_3_4": int(((lane["avg_position"] >= 3) & (lane["avg_position"] <= 4)).sum()),
        "missing_word_count": int(lane["word_count"].isna().sum()),
    },
    "monitoring_baseline": {
        "staleness_signal_verdict": "MIXED",
        "staleness_signal_n": 17,
        "honest_model_roc_auc_grouped_split": 0.727,
        "retrain_trigger_roc_auc_floor": 0.5,
    },
}
metrics_path = OUTPUTS_DIR / "w07_action_playbook_metrics.json"
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Wrote metrics receipt to: {metrics_path}")

fig, ax = plt.subplots(figsize=(6, 4))
action_counts = queue["action"].value_counts()
ax.bar(action_counts.index, action_counts.values, color=["#4C72B0", "#DD8452"])
ax.set_ylabel("Pages in queue")
ax.set_title("Action mix across the ranked queue")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
fig_path = FIGURES_DIR / "action_mix.png"
plt.savefig(fig_path, dpi=150)
plt.close()
print(f"Wrote figure to: {fig_path}")


Wrote ranked queue to: /home/claude/ML-Internship/work/outputs/action_playbook_queue.csv
Wrote metrics receipt to: /home/claude/ML-Internship/work/outputs/w07_action_playbook_metrics.json


Wrote figure to: /home/claude/ML-Internship/work/figures/action_mix.png


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.